# 力变大了，做功一定更多吗？

对应第十八章。只需要 NumPy、Matplotlib，从上到下运行即可。

我们研究恒定的一个力，在指定直线位移上做的功。其他力可能存在，但这里不把它们的功相加。图中英文标注避免中文字体缺失。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 11, "figure.dpi": 140})
BLUE, GREEN, ORANGE = "#2463a6", "#23845b", "#cb6b17"


## 1. 水平移动，增大竖直力

位移始终为 (3,0) 米。我们保持力的水平分量为 6 N，把竖直分量从 0 增加到 8 N。先预测：力的大小怎样变，做功又怎样变？

把下面 8 改成 20，再试一次。

In [ ]:
vertical_force = 8.0  # 再试 20
displacement = np.array([3.0, 0.0])
force_before = np.array([6.0, 0.0])
force_after = np.array([6.0, vertical_force])
for force in [force_before, force_after]:
    print("力 (N):", force, "大小 (N):", np.linalg.norm(force),
          "做功 (J):", force @ displacement)
assert np.isclose(force_before @ displacement, force_after @ displacement)

## 2. 保持力的大小，只改变方向

位移向右 3 米，力的大小始终为 10 N。先预测 60、90、120 度时，功分别是正、零还是负，再运行。

图中力以缩小后的箭头表示，它与位移有不同单位，不可以直接比较两种箭头长度。虚线表示力的水平分量。

In [ ]:
def draw_case(ax, angle_deg):
    angle = np.deg2rad(angle_deg)
    force = 10*np.array([np.cos(angle), np.sin(angle)])
    d = np.array([3.0, 0.0])
    work = float(force @ d)
    if abs(work) < 1e-10:
        work = 0.0
    scaled_force = 0.25*force
    ax.annotate("", xy=(3, 0), xytext=(0, 0),
                arrowprops=dict(arrowstyle="-|>", lw=3, color=GREEN))
    ax.annotate("", xy=scaled_force, xytext=(0, 0),
                arrowprops=dict(arrowstyle="-|>", lw=2.5, color=BLUE))
    ax.plot([0, scaled_force[0]], [0, 0], "--", lw=2, color=ORANGE)
    ax.plot([scaled_force[0]]*2, [0, scaled_force[1]], ":", color="0.5")
    ax.text(2.7, -0.45, "d = 3 m", color=GREEN, ha="right")
    ax.text(scaled_force[0], scaled_force[1]+0.23, "F = 10 N", color=BLUE, ha="center")
    ax.set(xlim=(-2.2, 3.4), ylim=(-0.8, 3.1), aspect="equal",
           title=f"{angle_deg:g} degrees: W = {work:g} J")
    ax.axhline(0, color="0.8", zorder=0)
    ax.axvline(0, color="0.8", zorder=0)
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)
    return work

fig, axes = plt.subplots(1, 3, figsize=(12, 3.7))
works = [draw_case(ax, angle) for ax, angle in zip(axes, [60, 90, 120])]
assert np.allclose(works, [15, 0, -15])
fig.tight_layout()
plt.show()

下面的角度可以自己修改。坐标相乘相加与“沿位移方向的力乘位移长度”应得到相同的结果。

In [ ]:
angle_deg = 60.0  # 再试 0、90、120、180
angle = np.deg2rad(angle_deg)
force = 10*np.array([np.cos(angle), np.sin(angle)])
d = np.array([3.0, 0.0])
coordinate_work = force @ d
parallel_work = np.linalg.norm(force)*np.linalg.norm(d)*np.cos(angle)
print("力的坐标 (N):", force)
print("坐标相乘相加 (J):", coordinate_work)
print("平行分量计算 (J):", parallel_work)
assert np.isclose(coordinate_work, parallel_work, atol=1e-10)
angles = np.linspace(0, 180, 181)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(angles, 30*np.cos(np.deg2rad(angles)), color=BLUE, lw=2.5)
ax.scatter([angle_deg], [coordinate_work], color=ORANGE, s=60, zorder=5)
ax.axhline(0, color="0.5", lw=0.8)
ax.set(xlim=(0, 180), xlabel="Angle between force and displacement (degrees)",
       ylabel="Work (J)", title="Same force magnitude, different work")
ax.set_xticks([0, 30, 60, 90, 120, 150, 180])
ax.grid(alpha=0.2)
fig.tight_layout()
plt.show()

## 3. 两个坐标都变化

把力设为 (6,8) N，位移设为 (3,4) m。先分别算水平与竖直分量的功，再求和。最后检查这个力是不是与位移同向。

试把位移换成 (-4,3)。两个坐标都有变化，为什么这个力做的功却是零？

In [ ]:
force = np.array([6.0, 8.0])
d = np.array([3.0, 4.0])  # 再试 [-4, 3]、[-3, -4]
component_work = force*d
work = float(component_work.sum())
cos_angle = (force @ d)/(np.linalg.norm(force)*np.linalg.norm(d))
print("两个坐标的贡献 (J):", component_work)
print("总和 (J):", work)
print("夹角 (度):", np.rad2deg(np.arccos(np.clip(cos_angle, -1, 1))))
assert np.isclose(work, force @ d)


## 从问题走到一般形式

我们把恒力做功写成 $F_xd_x+F_yd_y$。拿掉牛顿与米的具体含义，对两个同样长度的实向量定义 $u^Tv=\sum_i u_iv_i$，就是本书使用的标准内积。

用同一个运算，可以计算做功，也可以比较方向或求长度平方，但解释和单位不同。力与位移相乘给出焦耳；位移与自身相乘给出平方米。

合上实验：从水平位移出发，解释为什么垂直分量不做功，再写出任意方向的坐标公式。最后只用符号展开 $(u+v)^T(u+v)$，找出内积为零时的长度关系。

物理依据：[OpenStax：Work](https://openstax.org/books/university-physics-volume-1/pages/7-1-work)。